In [1]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

In [4]:
def get_exercise_type(row):
    tipos = [row.tipo_ejercicio_x, row.tipo_ejercicio_y]
    tipo  = [i for i in tipos if pd.notna(i)]
    if len(tipo) > 0:
        return tipo[0]
    else:
        return pd.NA

## Combinar enunciados con clasificaciones

Cargar enunciados

In [5]:
df_enun = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding="utf-8")

Establecer `id` como columna índice, para poder hacer `update`

In [6]:
df_enun = df_enun.set_index("id")

In [7]:
df_enun.head(2)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen,observaciones
id,,,,,,,,,,,
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_a,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,NaN,Discuta la veracidad de las siguientes afirmac...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...,NaN
fisica_campo_electrico_y_magnetico_2025_julio_ejercicio_b1,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,NaN,Dos conductores rectilíneos muy largos se disp...,FISICA. 2025. JULIO. EJERCICIO b1,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...,NaN


In [9]:
df_enun.columns

Index(['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
       'tipo_ejercicio', 'enunciado', 'info_ejercicio', 'archivo', 'imagen',
       'observaciones'],
      dtype='str')

## Cargar ejercicios clasificados **a incorporar**

In [10]:
df_classified = pd.read_csv(root / "csv/ejercicios_clasificados_configuracion_electronica.csv")

In [11]:
df_classified.head()

,tipo_ejercicio,confianza,id
0,Números cuánticos|Verdadero o Falso,alta,quimica_configuracion_electronica_2025_julio_e...
1,Situar elementos en tabla periódica (grupo y p...,alta,quimica_configuracion_electronica_2025_julio_e...
2,Situar elementos en tabla periódica (grupo y p...,media,quimica_configuracion_electronica_2025_junio_e...
3,Situar elementos en tabla periódica (grupo y p...,alta,quimica_configuracion_electronica_2025_reserva...
4,Combinaciones de números cuánticos|Configuraci...,alta,quimica_configuracion_electronica_2025_reserva...


Establecer `id` como columna índice, para poder hacer `update`

In [12]:
df_classified = df_classified.set_index("id")

In [13]:
df_classified.head(2)

,tipo_ejercicio,confianza
id,,
quimica_configuracion_electronica_2025_julio_ejercicio_1a,Números cuánticos|Verdadero o Falso,alta
quimica_configuracion_electronica_2025_julio_ejercicio_1b,Situar elementos en tabla periódica (grupo y p...,alta


In [14]:
df_classified.loc[:, ["asignatura", "tema"]] = ["Química", "configuración electrónica"]

In [15]:
df_classified[df_classified.tipo_ejercicio.isna()]

,tipo_ejercicio,confianza,asignatura,tema
id,,,,


Guardar cambios en el archivo (previo `reset_index()`)

In [16]:
df_classified.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados_configuracion_electronica.csv", index = False, encoding="utf-8")

### Inspeccionar clasificados

In [17]:
df_classified.columns

Index(['tipo_ejercicio', 'confianza', 'asignatura', 'tema'], dtype='str')

In [18]:
df_classified.sample()

,tipo_ejercicio,confianza,asignatura,tema
id,,,,
quimica_configuracion_electronica_2004_reserva_3_ejercicio_2_opcion_a,Situar elementos en tabla periódica (grupo y p...,alta,Química,configuración electrónica


Actualizar los ejercicios clasificados y resetear índice

In [19]:
df_enun.update(df_classified.tipo_ejercicio)
df_enun = df_enun.reset_index(drop = False)

In [20]:
rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','observaciones', 'enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
df_enun = df_enun[rearranged_cols]

Exportar enunciados clasificados actualizado

In [21]:
df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
               index = False, encoding="utf-8")

In [27]:
df_enun.query("asignatura == 'Química' and tema == 'configuración electrónica'").head(2)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,observaciones,enunciado,id,info_ejercicio,archivo,imagen
3335,Química,2025,Julio,Ejercicio 1A,configuración electrónica,Números cuánticos|Verdadero o Falso,NaN,Justifique la veracidad o falsedad de las sigu...,quimica_configuracion_electronica_2025_julio_e...,QUÍMICA. 2025. JULIO EJERCICIO 1A,source_pdf_files/Quimica/configuracion_electro...,enunciados_ejercicios/Quimica/configuracion_el...
3336,Química,2025,Julio,Ejercicio 1B,configuración electrónica,Situar elementos en tabla periódica (grupo y p...,NaN,"Sean los elementos A ( Z = 6 ) , B ( Z = 1 7 )...",quimica_configuracion_electronica_2025_julio_e...,QUÍMICA. 2025. JULIO EJERCICIO 1B,source_pdf_files/Quimica/configuracion_electro...,enunciados_ejercicios/Quimica/configuracion_el...


## Split del tipo de ejercicio
Ejercicios con más de un tipo de ejercicio

In [28]:
df_enun.shape

(4986, 12)

In [30]:
df_enun.tipo_ejercicio = df_enun.tipo_ejercicio.str.split("|")

In [31]:
df_enun = df_enun.explode("tipo_ejercicio")

In [58]:
df_enun.reset_index(drop = True, inplace=True)

In [59]:
df_enun.head()

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,observaciones,enunciado,id,info_ejercicio,archivo,imagen
0,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,NaN,NaN,Discuta la veracidad de las siguientes afirmac...,fisica_campo_electrico_y_magnetico_2025_julio_...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...
1,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,NaN,NaN,Dos conductores rectilíneos muy largos se disp...,fisica_campo_electrico_y_magnetico_2025_julio_...,FISICA. 2025. JULIO. EJERCICIO b1,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...
2,Física,2025,Julio,Ejercicio B2,campo eléctrico y magnético,NaN,NaN,Una carga q\n1\n= 2  1 0 − 9 C está fija en e...,fisica_campo_electrico_y_magnetico_2025_julio_...,FISICA. 2025. JULIO. EJERCICIO b2,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...
3,Física,2025,Junio,Ejercicio A,campo eléctrico y magnético,NaN,NaN,Dos partículas cargadas se mueven perpendicula...,fisica_campo_electrico_y_magnetico_2025_junio_...,FISICA. 2025. JUNIO. EJERCICIO a,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...
4,Física,2025,Junio,Ejercicio B1,campo eléctrico y magnético,NaN,NaN,El lado móvil de la espira rectangular de la f...,fisica_campo_electrico_y_magnetico_2025_junio_...,FISICA. 2025. JUNIO. EJERCICIO b1,source_pdf_files/Fisica/campo_electrico_magnet...,enunciados_ejercicios/Fisica/campo_electrico_m...


In [60]:
duplicated_indexes = df_enun[df_enun.duplicated(subset = ["id"])].index

In [61]:
duplicated_indexes

Index([3336, 3338, 3340, 3342, 3344, 3346, 3348, 3351, 3353, 3354,
       ...
       3646, 3649, 3651, 3654, 3658, 3660, 3661, 3662, 3666, 3667],
      dtype='int64', length=154)

In [62]:
df_enun[df_enun.tipo_ejercicio == "Conf. Electrónica"].tipo_ejercicio = "configuración electrónica"

In [63]:
renombrar_tipos = {'Verdadero o Falso' : "verdadero_falso",
 'Situar elementos en tabla periódica (grupo y período)' : "situar_tabla_periodica",
                    'Combinaciones de números cuánticos' : "combinaciones_numeros_cuanticos",
                             'Configuración electrónica' : "conf_electronica",
                                       'Ion más estable' : "ion_estable",
                    'Comparación energías de ionización' : "energias_ionizacion",
         'Configuración electrónica del ion más estable' : "conf_ion_estable",
                           'energía ionización Na+ y Ne' : "energia_ionizacion_na_ne",
}

In [64]:
df_enun.loc[duplicated_indexes[0]].tipo_ejercicio

'Verdadero o Falso'

In [65]:
for idx in duplicated_indexes:
    tipo_ejercicio = df_enun.loc[idx].tipo_ejercicio
    if tipo_ejercicio in renombrar_tipos.keys():
          add_id = remove_accents(renombrar_tipos[tipo_ejercicio]).lower()
    else:
        add_id = remove_accents(tipo_ejercicio).replace(" ", "_").lower()
    
    id = df_enun.loc[idx].id
    year_pos = id.find("2")
    new_id = id[:year_pos] + add_id + "_" + id[year_pos:]

    df_enun.loc[idx, "id"] = new_id  

In [71]:
mask = (df_enun.tema == "configuración electrónica") & (df_enun.tipo_ejercicio == "Conf. Electrónica")
df_enun.loc[mask, "tipo_ejercicio"] = "configuración electrónica"

In [72]:
df_enun.tipo_ejercicio = df_enun.tipo_ejercicio.str.lower()

In [73]:
df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
               index = False, encoding="utf-8")

Opcional: insertar enunciados limpios de matrices

In [208]:
df_clean_st = pd.read_csv(root / "classifier/matrices_limpio.csv")

In [209]:
df_clean_st.sample()

,id,enunciado
250,mates_ii_matrices_y_determinantes_2018_reserva...,Considera la matriz M = [matriz]. Sabiendo que...


In [212]:
df_enun = df_enun.set_index("id")
df_enun.update(df_clean_st.set_index("id").enunciado)

In [215]:
df_enun[df_enun.tema == "matrices y determinantes"].sample(3)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen
id,,,,,,,,,,
mates_ii_matrices_y_determinantes_2012_reserva_1_ejercicio_3_opcion_a,Mates II,2012,Reserva 1,"Ejercicio 3, opción A",matrices y determinantes,ecuación matricial,C onsidera las matrices: [matriz] A= [matriz] ...,MATEMÁTICAS II. 2012. RESERVA 1. EJERCICIO 3. ...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...
mates_ii_matrices_y_determinantes_2019_reserva_1_ejercicio_3_opcion_a,Mates II,2019,Reserva 1,"Ejercicio 3, opción A",matrices y determinantes,valores tenga inversa|ecuación matricial,Dadas las matrices A= m-1 m 0 y B = [matriz]\n...,MATEMÁTICAS II. 2019. RESERVA 1. EJERCICIO 3. ...,source_pdf_files/Mates II/matrices_determinant...,enunciados_ejercicios/Mates II/matrices_determ...
mates_ccss_matrices_y_determinantes_2006_reserva_2_ejercicio_1_opcion_a,Mates CCSS,2006,Reserva 2,"Ejercicio 1, opción A",matrices y determinantes,ecuación matricial,Sean las matricesA= [matriz] y B= [matriz]. Ca...,SOCIALES II. 2006 RESERVA 2. EJERCICIO 1. OPCI...,source_pdf_files/Mates CCSS/matrices_determina...,enunciados_ejercicios/Mates CCSS/matrices_dete...


Exportar enunciados clasificados actualizado

In [216]:
df_enun.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados.csv",
                                         index = False, encoding="utf-8")